# 11 — Product Metadata Retrieval Debug

نسخه‌ی جدید علاوه بر RRF، **product-type/category grounding** دارد.

ستون‌های کلیدی:
- `token_overlap`
- `title_bigram_overlap`
- `category_score`
- `rrf_score`
- `metadata_score`

دو failure هدف:
1. تونیک در query شامپو
2. شورت گنی در query کرم آبرسان


In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.rag.config import load_config
from src.rag.embedding.factory import EmbeddingFactory
from src.rag.preprocessing.processor import TextProcessor
from src.rag.product_search import (
    ProductFAISSIndex,
    ProductBM25Index,
    ProductMetadataRetriever,
)

In [2]:
rag_config = load_config(
    PROJECT_ROOT
    / "configs"
    / "rag.yaml"
)

search_config = load_config(
    PROJECT_ROOT
    / "configs"
    / "product_search.yaml"
)

metadata_cfg = (
    search_config[
        "product_search"
    ][
        "metadata"
    ]
)

processor = TextProcessor()

embedding_model = (
    EmbeddingFactory.create(
        provider=rag_config[
            "embedding"
        ][
            "provider"
        ],
        model_name=rag_config[
            "embedding"
        ][
            "model"
        ],
    )
)

dense_index = (
    ProductFAISSIndex()
    .load(
        PROJECT_ROOT
        / "data"
        / "indexes"
        / "products_embedding"
    )
)

sparse_index = (
    ProductBM25Index(
        processor=processor
    )
    .load(
        PROJECT_ROOT
        / "data"
        / "indexes"
        / "products_bm25_tantivy"
    )
)

retriever = ProductMetadataRetriever(
    embedding_model=embedding_model,
    dense_index=dense_index,
    sparse_index=sparse_index,
    processor=processor,
    bm25_weight=metadata_cfg[
        "bm25_weight"
    ],
    embedding_weight=metadata_cfg[
        "embedding_weight"
    ],
    candidate_multiplier=metadata_cfg[
        "candidate_multiplier"
    ],
    brand_boost=metadata_cfg[
        "brand_boost"
    ],
    lexical_weight=metadata_cfg[
        "lexical_weight"
    ],
    rrf_k=metadata_cfg[
        "rrf_k"
    ],
    validate_index_alignment=metadata_cfg[
        "validate_index_alignment"
    ],
)

print("Product index alignment: OK")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Product index alignment: OK


In [3]:
queries = [
    "ضد آفتاب پوست چرب که جوش نزنه",
    "شامپو ضد ریزش سریتا",
    "کرم آبرسان سبک و زود جذب",
]

debug_columns = [
    "id",
    "title_fa",
    "Brand",
    "Category2",
    "metadata_score",
    "rrf_score",
    "lexical_score",
    "token_overlap",
    "title_bigram_overlap",
    "category_score",
    "bm25_rank",
    "embedding_rank",
    "bm25_raw_score",
    "embedding_raw_score",
    "brand_match",
]

for query in queries:
    print()
    print("=" * 100)
    print("QUERY:", query)

    results = retriever.retrieve(
        query,
        top_k=20,
    )

    display(
        results[
            debug_columns
        ]
    )


QUERY: ضد آفتاب پوست چرب که جوش نزنه


,id,title_fa,Brand,Category2,metadata_score,rrf_score,lexical_score,token_overlap,title_bigram_overlap,category_score,bm25_rank,embedding_rank,bm25_raw_score,embedding_raw_score,brand_match
0,1110394,کرم ضد آفتاب رنگی اریکه مدل SPF50 مناسب پوست ه...,اریکه,کرم ضد آفتاب,0.673892,0.823153,0.45,0.666667,0.2,0.4,4.0,28.0,32.729515,0.618812,0.0
1,10985328,فلوئید ضد آفتاب بی رنگ نوکس SPF50 مدل LIGHT من...,نوکس,کرم ضد آفتاب,0.673892,0.823153,0.45,0.666667,0.2,0.4,28.0,4.0,30.930351,0.652808,0.0
2,11286406,فلوئید ضد آفتاب بی رنگ اوریاژ +SPF50 مدل Hysea...,اوریاژ,کرم ضد آفتاب,0.654736,0.791226,0.45,0.666667,0.2,0.4,29.0,8.0,30.930351,0.646624,0.0
3,9156746,کرم ضد آفتاب رنگی بیکن +SPF50 شماره +1 مناسب پ...,بیکن,کرم ضد آفتاب,0.636684,0.761141,0.45,0.666667,0.2,0.4,6.0,42.0,32.194176,0.608636,0.0
4,6733478,فلوئید ضد آفتاب بی رنگ الارو SPF50 مدل Ultra L...,الارو,کرم ضد آفتاب,0.623636,0.739394,0.45,0.666667,0.2,0.4,50.0,6.0,30.452507,0.651996,0.0
5,5536401,کرم ضد آفتاب رنگی سینره +SPF60 مدل 8411 مناسب ...,سینره,کرم ضد آفتاب,0.623317,0.738861,0.45,0.666667,0.2,0.4,13.0,35.0,32.194176,0.611982,0.0
6,12243791,کرم ضد آفتاب بدون رنگ SPF 30 تینولا مدل Tino S...,متفرقه,کرم ضد آفتاب,0.614571,0.724284,0.45,0.666667,0.2,0.4,67.0,3.0,28.446892,0.657175,0.0
7,183277,کرم ضد آفتاب رنگی الارو SPF50 مدل Teinte Fonce...,الارو,کرم ضد آفتاب,0.605524,0.709207,0.45,0.666667,0.2,0.4,20.0,33.0,31.933105,0.612756,0.0
8,12390858,کرم ضد آفتاب بدون رنگ سیلوژه SPF 50 مدل TEXTUR...,سیلوژه,کرم ضد آفتاب,0.592798,0.687997,0.45,0.666667,0.2,0.4,64.0,9.0,28.938171,0.644173,0.0
9,12550788,کرم ضد آفتاب رنگی اریکه SPF 50 مدل Solarie And...,اریکه,کرم ضد آفتاب,0.569158,0.648596,0.45,0.666667,0.2,0.4,69.0,14.0,27.972605,0.635154,0.0



QUERY: شامپو ضد ریزش سریتا


,id,title_fa,Brand,Category2,metadata_score,rrf_score,lexical_score,token_overlap,title_bigram_overlap,category_score,bm25_rank,embedding_rank,bm25_raw_score,embedding_raw_score,brand_match
0,711943,شامپو ضد ریزش سریتا مدل پرومین مناسب برای موها...,سریتا,شامپو مو,0.706667,0.484127,0.857143,1.00,1.000000,0.285714,3.0,NaN,35.896019,NaN,1.0
1,6552521,شامپو تقویت کننده و ضد ریزش سریتا مدل Fortifyi...,سریتا,شامپو مو,0.676952,0.500000,0.740476,1.00,0.666667,0.285714,1.0,NaN,36.173233,NaN,1.0
2,8021274,شامپو ضد ریزش مو سریتا مدل مینوتا حجم 200 میلی...,سریتا,شامپو مو,0.662183,0.469231,0.740476,1.00,0.666667,0.285714,5.0,NaN,35.571182,NaN,1.0
3,3832629,شامپو ضد ریزش مو سریتا مدل مینوتا مناسب برای م...,سریتا,شامپو مو,0.655460,0.455224,0.740476,1.00,0.666667,0.285714,7.0,NaN,34.989395,NaN,1.0
4,1147482,شامپو ضد ریزش مو فورکاپیل مدل procapil حجم 200...,فورکاپیل,شامپو مو,0.650213,0.665038,0.627976,0.75,0.666667,0.285714,73.0,10.0,23.428490,0.696847,0.0
5,471187,شامپو تقویت کننده و ضد ریزش سریتا مدل کافئین م...,سریتا,شامپو مو,0.649126,0.442029,0.740476,1.00,0.666667,0.285714,9.0,NaN,33.882515,NaN,1.0
6,296253,شامپو ضد ریزش مو مورست حجم 250 میلی لیتر,مورست,شامپو مو,0.649064,0.663122,0.627976,0.75,0.666667,0.285714,31.0,33.0,24.325674,0.662043,0.0
7,10324133,شامپو ضد ریزش مو گلیس مدل NUTRIBAL حجم 500 میل...,گلیس,شامپو مو,0.637643,0.644087,0.627976,0.75,0.666667,0.285714,54.0,21.0,23.428490,0.675327,0.0
8,471143,شامپو تقویت کننده و ضد ریزش سریتا مدل فورتی فا...,سریتا,شامپو مو,0.637500,0.417808,0.740476,1.00,0.666667,0.285714,13.0,NaN,33.355579,NaN,1.0
9,476822,تونیک ضد ریزش مو سریتا مدل پرومین حجم 60 میلی ...,سریتا,شامپو مو,0.635748,0.491935,0.623810,1.00,0.333333,0.285714,2.0,NaN,35.905472,NaN,1.0



QUERY: کرم آبرسان سبک و زود جذب


,id,title_fa,Brand,Category2,metadata_score,rrf_score,lexical_score,token_overlap,title_bigram_overlap,category_score,bm25_rank,embedding_rank,bm25_raw_score,embedding_raw_score,brand_match
0,9585339,کرم آبرسان ناک مدل حاوی پروپولیس,ناک,کرم مرطوب کننده و نرم کننده,0.600211,0.795351,0.3075,0.4,0.25,0.2,3.0,38.0,17.801384,0.658882,0.0
1,9284669,کرم آبرسان سیمپل مدل Light حجم 125 میلی لیتر,سیمپل,کرم مرطوب کننده و نرم کننده,0.489755,0.611258,0.3075,0.4,0.25,0.2,57.0,27.0,16.745489,0.667794,0.0
2,3347106,کرم آبرسان پانی دراگ مدل Light حجم 150 میلی لیتر,پانی دراگ,کرم مرطوب کننده و نرم کننده,0.423000,0.500000,0.3075,0.4,0.25,0.2,NaN,1.0,NaN,0.732426,0.0
3,6282840,کرم آبرسان زیکسار مدل آبرسان و ضد چروک حجم 50 ...,زیکسار,کرم مرطوب کننده و نرم کننده,0.418161,0.491935,0.3075,0.4,0.25,0.2,2.0,NaN,18.575668,NaN,0.0
4,6556015,کرم آبرسان نوتروژنا مدل Besleyiciحجم 200میلی لیتر,نوتروژینا,کرم مرطوب کننده و نرم کننده,0.404538,0.469231,0.3075,0.4,0.25,0.2,5.0,NaN,17.434475,NaN,0.0
5,4427742,کرم آبرسان کامان سری واتربمب مدل 003 حجم 200 م...,کامان,کرم مرطوب کننده و نرم کننده,0.400273,0.462121,0.3075,0.4,0.25,0.2,6.0,NaN,17.429705,NaN,0.0
6,4428092,کرم آبرسان کامان سری واتربمب مدل 002 حجم 200 م...,کامان,کرم مرطوب کننده و نرم کننده,0.396134,0.455224,0.3075,0.4,0.25,0.2,7.0,NaN,17.429705,NaN,0.0
7,4428508,کرم آبرسان کامان سری واتربمب مدل 001 حجم 200 م...,کامان,کرم مرطوب کننده و نرم کننده,0.392118,0.448529,0.3075,0.4,0.25,0.2,8.0,NaN,17.429705,NaN,0.0
8,5927558,کرم آبرسان کامان مدل Collagen حجم 500 میلی لیت...,کامان,کرم مرطوب کننده و نرم کننده,0.388217,0.442029,0.3075,0.4,0.25,0.2,9.0,NaN,17.429705,NaN,0.0
9,10530170,کرم آبرسان فریدن مدل 01 حجم 50 میلی لیتر,فریدن,کرم مرطوب کننده و نرم کننده,0.386323,0.438872,0.3075,0.4,0.25,0.2,80.0,78.0,16.745489,0.640512,0.0
